# 01 · Ingesta y perfilado

Perfilado completo, independiente del estado de 00. No se limpian, imputan ni eliminan registros. No se desarrolla EDA.

In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "AGENTS.md").exists() and (root.parent / "AGENTS.md").exists():
    root = root.parent
assert (root / "AGENTS.md").exists(), "Ejecutar desde raíz o notebooks/"
sys.path.insert(0, str(root))
import pandas as pd
from IPython.display import display
from src.data.config import REPORTS
pd.set_option("display.max_rows", 100)


## 1. Ingesta estricta y perfilado completo

CSV UTF-8 con BOM admitido, delimitador coma y comillas dobles. Errores estructurales detienen la ejecución. Nulos, cardinalidades, duplicados y relaciones se cuentan sobre todos los registros. Duplicados de fila excluyen la ruta del archivo.

Casi constante: valor dominante ≥99% de los no nulos. Tipo numérico candidato: ≥95% de no nulos convertibles (excepto IDs/códigos/números identificadores). Fechas candidatas según nombres observados; se valida conversión a TIMESTAMP. Posibles outliers: regla 1,5×IQR sobre números finitos, cuartiles exactos; no equivale a error y es sensible a mezclas de monedas y escalas. Corte de fechas futuras: 2026-09-28.

In [2]:
from src.data.eda.profiling_utils import profiling
columns, datasets, keys, relations = profiling()
profile = pd.DataFrame(columns)
display(pd.DataFrame(datasets))

Profiling: branches


Profiling: call_center_interactions


Profiling: call_transcripts


Profiling: campaign_sends


Profiling: complaints


...

Profiling: products


Profiling: satisfaction_surveys


Profiling: service_agents


Profiling: transactions


,dataset,rows,duplicate_rows_excess
0,branches,350,0
1,call_center_interactions,686296,0
2,call_transcripts,171321,0
3,campaign_sends,1746801,0
4,complaints,67095,0
5,customers,150000,0
6,daily_exchange_rates,13164,0
7,digital_events,15620994,0
8,marketing_campaigns,200,0
9,products,400000,0


## 2. Nulos, tipos, cardinalidad y constantes

In [3]:
display(profile[['dataset','column','inferred_type','nulls','null_pct','cardinality','constant','almost_constant']])

,dataset,column,inferred_type,nulls,null_pct,cardinality,constant,almost_constant
0,branches,branch_id,TEXT,0,0.000000,350,False,False
1,branches,branch_code,TEXT,0,0.000000,350,False,False
2,branches,branch_name,TEXT,0,0.000000,175,False,False
3,branches,branch_type,TEXT,0,0.000000,4,False,False
4,branches,address,TEXT,0,0.000000,350,False,False
...,...,...,...,...,...,...,...,...
255,transactions,response_code,TEXT,221033,4.995087,5,False,False
256,transactions,is_fraud,BOOLEAN,0,0.000000,2,False,True
257,transactions,fraud_score,NUMERIC_CANDIDATE,885157,20.003512,5014,False,False
258,transactions,latitude,NUMERIC_CANDIDATE,3567680,80.625391,850555,False,False


## 3. Fechas, rangos y posibles outliers

In [4]:
display(profile[profile.possible_date][['dataset','column','invalid_dates','future_dates','minimum','maximum']])
display(profile[profile.inferred_type == 'NUMERIC_CANDIDATE'][['dataset','column','minimum','maximum','numeric_parse_failures','non_finite','q1','q3','possible_outliers']])

,dataset,column,invalid_dates,future_dates,minimum,maximum
20,branches,branch_opening_date,0.0,0.0,1990-01-03 00:00:00,2023-05-11 00:00:00
23,call_center_interactions,interaction_date,0.0,0.0,2023-06-17 08:03:26,2026-06-18 07:58:13
24,call_center_interactions,process_date,0.0,0.0,2023-06-17 00:00:00,2026-06-17 00:00:00
45,call_transcripts,process_date,0.0,0.0,2023-06-17 00:00:00,2026-06-17 00:00:00
62,campaign_sends,send_date,0.0,0.0,2023-07-01 06:00:42,2026-06-18 05:59:53
63,campaign_sends,process_date,0.0,0.0,2023-07-01 00:00:00,2026-06-17 00:00:00
72,campaign_sends,open_date,0.0,0.0,2023-07-01 07:44:22,2026-06-25 03:30:02
74,campaign_sends,click_date,0.0,0.0,2023-07-01 21:57:26,2026-06-24 23:51:59
77,campaign_sends,conversion_date,0.0,0.0,2023-07-03 12:56:27,2026-06-26 08:26:49
84,complaints,creation_date,0.0,0.0,2023-06-17 08:07:05,2026-06-18 07:56:52


,dataset,column,minimum,maximum,numeric_parse_failures,non_finite,q1,q3,possible_outliers
15,branches,atm_count,2.0,8.0,0.0,0.0,3.000000,7.000000e+00,0.0
17,branches,teller_window_count,3.0,12.0,0.0,0.0,6.000000,1.000000e+01,0.0
18,branches,latitude,-34.690767,25.781772,0.0,0.0,-0.043213,6.334851e+00,125.0
19,branches,longitude,-103.44855,0.099995,0.0,0.0,-76.626129,-9.045450e-03,0.0
31,call_center_interactions,duration_seconds,30.0,1204.0,0.0,0.0,205.000000,4.150000e+02,8612.0
32,call_center_interactions,wait_time_seconds,0.0,424.0,0.0,0.0,79.000000,1.600000e+02,1669.0
36,call_center_interactions,sentiment_score,-1.0,1.0,0.0,0.0,-0.250000,2.000000e-01,17229.0
53,call_transcripts,accent_confidence,0.75,0.99,0.0,0.0,0.810000,9.300000e-01,0.0
60,call_transcripts,duration_seconds,30.0,1151.0,0.0,0.0,205.000000,4.150000e+02,2186.0
75,campaign_sends,click_count,1.0,5.0,0.0,0.0,2.000000,4.000000e+00,0.0


## 4. Calidad de IDs y claves candidatas

In [5]:
display(pd.DataFrame(keys))

,dataset,columns,null_rows,distinct,duplicate_non_null_excess,candidate_key,scope
0,branches,branch_id,0,350,0,True,single_column
1,branches,branch_code,0,350,0,True,single_column
2,branches,address,0,350,0,True,single_column
3,branches,phone,0,350,0,True,single_column
4,branches,email,0,350,0,True,single_column
5,branches,latitude,0,350,0,True,single_column
6,branches,longitude,0,350,0,True,single_column
7,call_center_interactions,interaction_id,0,686296,0,True,single_column
8,call_center_interactions,customer_id,0,148443,537853,False,single_column
9,call_center_interactions,agent_id,0,1090,685206,False,single_column


## 5. Relaciones: cobertura e integridad de referencias

Las comparaciones usan valores originales exactos, sin trim ni normalización. La cobertura valida coincidencias, no semántica. Un destino con IDs repetidos puede multiplicar filas en un join. Las claves compuestas se prueban solo donde existe una hipótesis explícita: fecha y par de monedas en tipos de cambio.

In [6]:
display(pd.DataFrame(relations))

,source_dataset,source_column,target_dataset,target_column,source_rows,null_rows,matched_rows,orphan_rows,match_pct_non_null,parent_duplicate_excess,status
0,call_center_interactions,customer_id,customers,customer_id,686296,0,686296,0,100.000000,0,"hypothesis; value overlap checked, business me..."
1,call_center_interactions,agent_id,service_agents,agent_id,686296,0,686296,0,100.000000,0,"hypothesis; value overlap checked, business me..."
2,call_transcripts,interaction_id,call_center_interactions,interaction_id,171321,0,171321,0,100.000000,0,"hypothesis; value overlap checked, business me..."
3,call_transcripts,customer_id,customers,customer_id,171321,0,171321,0,100.000000,0,"hypothesis; value overlap checked, business me..."
4,call_transcripts,agent_id,service_agents,agent_id,171321,0,171321,0,100.000000,0,"hypothesis; value overlap checked, business me..."
5,campaign_sends,campaign_id,marketing_campaigns,campaign_id,1746801,0,1746801,0,100.000000,0,"hypothesis; value overlap checked, business me..."
6,campaign_sends,customer_id,customers,customer_id,1746801,0,1746801,0,100.000000,0,"hypothesis; value overlap checked, business me..."
7,complaints,customer_id,customers,customer_id,67095,0,67095,0,100.000000,0,"hypothesis; value overlap checked, business me..."
8,complaints,affected_product_id,products,product_id,67095,22525,44570,0,100.000000,0,"hypothesis; value overlap checked, business me..."
9,complaints,related_branch_id,branches,branch_id,67095,47917,19178,0,100.000000,0,"hypothesis; value overlap checked, business me..."


## 6. Hallazgos y limitaciones

Consultar column_profiles.csv, duplicates.csv, candidate_keys.csv y relationships.csv. Los nulos pueden ser estructurales por tipo de evento/producto y requieren interpretación posterior. IDs iguales no garantizan que cliente, producto o agente coincidan entre tablas. No se evalúan todas las reglas de negocio, coherencia temporal entre eventos ni todas las combinaciones de claves. Los valores atípicos se registran sin corregir. No se generan gráficos ni análisis de comportamiento.

In [7]:
import json
display(json.loads((REPORTS / 'profiling_integrity.json').read_text()))
print('Columnas totalmente nulas:', int((profile.nulls == profile.rows).sum()))
print('Columnas constantes:', int(profile.constant.sum()))
print('Columnas casi constantes:', int(profile.almost_constant.sum()))

{'files_verified': 7671, 'sha256_unchanged': True}

Columnas totalmente nulas: 1
Columnas constantes: 7
Columnas casi constantes: 2
